# B1 — Zero-shot NLI

`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`, no training. Three hypothesis wordings (Arabic / French / English),
every preprocessing of `results.csv` (+ a few PLM-oriented ones). `P(sarcastic)` is thresholded on **val**. Test stays sealed.

In [ ]:
# Colab only: packages missing from the default image
%pip install -q emoji tashaphyne arabert pyarabic sentencepiece protobuf

In [ ]:
import os, sys

PROJECT = "/content/drive/MyDrive/tp2-arabic-sarcasm"   # <- where the repo lives in your Drive
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT + "/notebooks")
sys.path.append("..")

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from transformers import pipeline

from src.data import get_data
from src.evaluate import log_result, metrics, plot_results, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, set_seed

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
train, val, test, report = get_data()
y_val = val["sarcasm"].values
print(f"val: {len(val)} | sarcasm rate: {y_val.mean():.3f}")

In [ ]:
BASE = ["html", "mentions", "hashtags"]
EMOJI = ["emoji"]
NORM = ["diacritics", "kashida", "letters", "elongation"]
LIGHT = ["light_stem"]
URLS = ["urls"]

COMBOS = {
    # same names / definitions as 02_baseline (so rows are comparable in results.csv)
    "base":                      BASE,
    "emoji":                     BASE + EMOJI,
    "norm":                      BASE + NORM,
    "light_stem":                BASE + LIGHT,
    "emoji + norm":              BASE + EMOJI + NORM,
    "emoji + light_stem":        BASE + EMOJI + LIGHT,
    "norm + light_stem":         BASE + NORM + LIGHT,
    "emoji + norm + light_stem": BASE + EMOJI + NORM + LIGHT,
    "urls kept":                 BASE,
    # added for PLMs: URLs -> <URL>, no stemming, AraBERT's own preprocessor
    "urls replaced":             BASE + URLS,
    "emoji + urls replaced":     BASE + URLS + EMOJI,
    "minimal":                   ["html", "urls", "mentions"],   # what 05_finetune uses
    "arabert":                   None,
}

In [ ]:
def make_prep(steps):
    if steps is None:
        from arabert.preprocess import ArabertPreprocessor
        return ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter").preprocess
    return build_pipeline(steps)


def prepare(frame, steps):
    prep = make_prep(steps)
    texts = [prep(tweet) for tweet in frame["tweet"]]
    return [t if t.strip() else raw for t, raw in zip(texts, frame["tweet"])]


try:
    make_prep(None)
except Exception as error:
    print("arabert preprocessing skipped:", error)
    COMBOS.pop("arabert")

print(list(COMBOS))
print("base       :", prepare(val.head(4), COMBOS["base"])[3][:100])
print("urls kept == base (BASE has no urls step), so those two rows are expected to match")

In [ ]:
def slug(name):
    return name.replace(" + ", "+").replace(" ", "_")

HYPOTHESES = {
    "ar": (["ساخرة", "جادة"],          "هذه التغريدة {}."),
    "fr": (["sarcastique", "sérieux"], "Ce tweet est {}."),
    "en": (["sarcastic", "serious"],   "This tweet is {}."),
}

zs = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=0 if DEVICE == "cuda" else -1,
)

In [ ]:
def run(combos, lang):
    labels, template = HYPOTHESES[lang]
    name_model = f"zeroshot_nli_{lang}"
    rows, seen = [], {}

    for name, steps in combos.items():
        texts = prepare(val, steps)
        h = hash(tuple(texts))

        start = time.time()
        if h not in seen:                       # identical texts (base / urls kept) are scored once
            out = zs(texts, candidate_labels=labels, hypothesis_template=template, batch_size=32)
            seen[h] = np.array([dict(zip(o["labels"], o["scores"]))[labels[0]] for o in out])
        proba = seen[h]
        seconds = time.time() - start

        threshold = best_threshold(y_val, proba)
        scores = metrics(y_val, proba, threshold)

        log_result(
            model=name_model,
            preprocessing=name,
            seed=42,
            threshold=threshold,
            train_time=0.0,
            trained_params=0,
            **{f"val_{k}": v for k, v in scores.items()}
        )
        save_predictions(f"{name_model}_{slug(name)}", 42, val["id"], y_val, proba, threshold)

        show(f"{lang} | {name}", scores, threshold, seconds)
        rows.append({"model": name_model, "preprocessing": name, "threshold": threshold, **scores})

    return pd.DataFrame(rows).set_index("preprocessing")

In [ ]:
tables = {lang: run(COMBOS, lang) for lang in HYPOTHESES}

In [ ]:
print(f"val F1 (sarcastic) — a random classifier has PR-AUC ≈ {y_val.mean():.3f}")
pd.DataFrame({lang: t["f1"] for lang, t in tables.items()}).round(4)

In [ ]:
pd.DataFrame({lang: t["pr_auc"] for lang, t in tables.items()}).round(4)

In [ ]:
summary = pd.concat([t.assign(name=lang + " | " + t.index) for lang, t in tables.items()]).set_index("name")
plot_results(summary.nlargest(15, "f1"))